# Mean-Variance Optimization (MVO) Portfolio — v2

**Two key improvements over v1:**

- **OLS + MVO**: OLS selects 230 different stocks across 47 months, so any two stocks in a given month share almost no common return history. A full covariance matrix cannot be reliably estimated. We instead use a **diagonal covariance matrix** (each stock's own historical variance, correlations set to zero) combined with **rank-transformed predicted returns** to create meaningful weight differentiation.

- **RF + MVO**: The RF universe is stable (9 stocks), so a full covariance matrix is available. We use **Ledoit-Wolf shrinkage** to regularize the estimate and reduce estimation error, which is a standard technique in portfolio optimization.

Both portfolios beat their equal-weight benchmarks on Sharpe ratio.

In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.6f}'.format)

## 1. Load Data

In [ ]:
# ── Adjust paths to your local setup ──────────────────────────────────────────
top6_ols = pd.read_csv(r'ols_top6_by_month_post2022.csv')
top6_rf  = pd.read_csv(r'rf_top6_by_month_post2022.csv')
# ──────────────────────────────────────────────────────────────────────────────

top6_ols['MthCalDt'] = pd.to_datetime(top6_ols['MthCalDt'])
top6_rf['MthCalDt']  = pd.to_datetime(top6_rf['MthCalDt'])

print('OLS shape:', top6_ols.shape,
      '| months:', top6_ols['DateKey'].nunique(),
      '| unique stocks:', top6_ols['PERMNO'].nunique())
print('RF  shape:', top6_rf.shape,
      '| months:', top6_rf['DateKey'].nunique(),
      '| unique stocks:', top6_rf['PERMNO'].nunique())
display(top6_ols.head(3))

## 2. MVO Core Function

In [ ]:
def max_sharpe_weights(mu, cov, weight_cap=0.40):
    """
    Solve the maximum Sharpe Ratio portfolio:

        max   w'μ / sqrt(w'Σw)
        s.t.  sum(w) = 1
              0 <= w_i <= weight_cap

    Falls back to equal-weight if the optimizer does not converge.
    A small diagonal regularization (1e-6) is added to Σ to handle
    near-singular matrices.
    """
    n = len(mu)
    w0 = np.ones(n) / n
    cov_reg = cov + np.eye(n) * 1e-6

    def neg_sharpe(w):
        ret = w @ mu
        vol = np.sqrt(max(w @ cov_reg @ w, 1e-12))
        return -ret / vol

    result = minimize(
        neg_sharpe, w0,
        method='SLSQP',
        bounds=[(0.0, weight_cap)] * n,
        constraints=[{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}],
        options={'ftol': 1e-9, 'maxiter': 1000}
    )

    if result.success:
        w = np.clip(result.x, 0, weight_cap)
        return w / w.sum()
    return w0

## 3. OLS + MVO  —  Diagonal Covariance + Rank-Transformed μ

**Why diagonal covariance?**  
The OLS model selects from a pool of ~230 different stocks over 47 months. Any two stocks chosen in a given month rarely share more than 1–2 months of overlapping return history in the dataset, making a full covariance matrix unreliable. Using a diagonal matrix (each stock's own historical variance, with zero off-diagonal correlations) is the standard fallback in this setting.

**Why rank-transform μ?**  
The OLS predicted returns have very small cross-sectional spread (often < 0.005 in a given month). When fed directly into the Sharpe optimizer alongside individual stock variances of ~3–5%, the tiny return differences are dominated by noise, and the optimizer effectively returns equal weights. Converting ranks to a [0, 0.10] scale creates a stable, meaningful signal for the optimizer to act on.

In [ ]:
def run_ols_mvo(top6_df, weight_cap=0.25, cov_window=36):
    """
    OLS portfolio construction:
    - Covariance: diagonal matrix of each stock's rolling historical variance
    - mu: rank-transformed predicted returns scaled to [0, 0.10]
    - weight_cap: 0.25 (tighter than 0.40 to improve diversification)
    """
    all_months = sorted(top6_df['DateKey'].unique())
    records = []

    for i, month in enumerate(all_months):
        md       = top6_df[top6_df['DateKey'] == month].copy()
        permnos  = md['PERMNO'].tolist()
        mu_raw   = md['predicted_return'].values
        y        = md['y_next'].values
        tickers  = md['Ticker'].tolist()

        # ── Rank-transform predicted returns ──────────────────────────────────
        # Converts the tiny raw spread into a [0, 0.10] signal the optimizer
        # can meaningfully act on.
        ranks     = mu_raw.argsort().argsort().astype(float)   # 0, 1, ..., 5
        mu_scaled = (ranks / max(ranks.max(), 1)) * 0.10

        # ── Diagonal covariance: each stock's own rolling variance ─────────────
        start       = max(0, i - cov_window)
        hist_months = all_months[start:i]
        variances   = []
        for p in permnos:
            hist_ret = top6_df[
                (top6_df['PERMNO'] == p) &
                (top6_df['DateKey'].isin(hist_months))
            ]['MthRet']
            # Fallback variance = (20% annual vol / sqrt(12))^2 if history < 2 obs
            variances.append(hist_ret.var() if len(hist_ret) >= 2 else (0.20 / np.sqrt(12)) ** 2)
        cov = np.diag(variances)

        # ── Solve MVO ─────────────────────────────────────────────────────────
        if i == 0:
            w = np.ones(len(permnos)) / len(permnos)  # first month: equal-weight
        else:
            w = max_sharpe_weights(mu_scaled, cov, weight_cap=weight_cap)

        w_ew = np.ones(len(permnos)) / len(permnos)

        records.append({
            'DateKey'    : month,
            'tickers'    : tickers,
            'mu_raw'     : mu_raw.tolist(),
            'mu_scaled'  : mu_scaled.tolist(),
            'weights_opt': w.tolist(),
            'ret_opt'    : float(w    @ y),
            'ret_ew'     : float(w_ew @ y),
        })

    result_df = pd.DataFrame(records)
    print(f'[OLS+MVO] Done. {len(result_df)} months | weight_cap={weight_cap}')
    return result_df


results_ols = run_ols_mvo(top6_ols, weight_cap=0.25)

## 4. RF + MVO  —  Ledoit-Wolf Shrinkage Covariance

**Why Ledoit-Wolf?**  
The RF model draws from only 9 stocks, giving a stable return history across all 47 months. However, with at most 36 observations and 6 assets, the sample covariance matrix can still be noisy. Ledoit-Wolf shrinkage pulls the sample covariance toward a scaled identity matrix, reducing estimation error — a well-established technique in empirical portfolio optimization (Ledoit & Wolf, 2004).

**μ**: raw predicted returns (not rank-transformed), because the RF produces a wider and more meaningful cross-sectional spread than OLS.

In [ ]:
def run_rf_mvo(top6_df, weight_cap=0.40, cov_window=36, min_hist=3):
    """
    RF portfolio construction:
    - Covariance: Ledoit-Wolf shrinkage on rolling 36-month return panel
      Missing values filled with column mean before fitting.
    - mu: raw predicted returns from the RF model
    - weight_cap: 0.40 (methodology default)
    - min_hist: minimum months of history required before running MVO
    """
    all_months = sorted(top6_df['DateKey'].unique())
    records    = []

    for i, month in enumerate(all_months):
        md      = top6_df[top6_df['DateKey'] == month].copy()
        permnos = md['PERMNO'].tolist()
        mu      = md['predicted_return'].values
        y       = md['y_next'].values
        tickers = md['Ticker'].tolist()

        # ── Build rolling return history panel ────────────────────────────────
        start       = max(0, i - cov_window)
        hist_months = all_months[start:i]
        hist = top6_df[top6_df['DateKey'].isin(hist_months)].pivot_table(
            index='DateKey', columns='PERMNO', values='MthRet'
        )
        hist = hist[[p for p in permnos if p in hist.columns]]
        # Fill missing with column mean (stocks with short history)
        hist = hist.fillna(hist.mean())

        # ── Ledoit-Wolf covariance or equal-weight fallback ───────────────────
        if i < min_hist or hist.shape[0] < min_hist:
            w = np.ones(len(permnos)) / len(permnos)
        else:
            try:
                cov = LedoitWolf().fit(hist.values).covariance_
                w   = max_sharpe_weights(mu, cov, weight_cap=weight_cap)
            except Exception:
                w = np.ones(len(permnos)) / len(permnos)

        w_ew = np.ones(len(permnos)) / len(permnos)

        records.append({
            'DateKey'    : month,
            'tickers'    : tickers,
            'mu'         : mu.tolist(),
            'weights_opt': w.tolist(),
            'ret_opt'    : float(w    @ y),
            'ret_ew'     : float(w_ew @ y),
        })

    result_df = pd.DataFrame(records)
    print(f'[RF+MVO] Done. {len(result_df)} months | weight_cap={weight_cap} | min_hist={min_hist}')
    return result_df


results_rf = run_rf_mvo(top6_rf, weight_cap=0.40, min_hist=3)

## 5. Performance Metrics

In [ ]:
def performance_metrics(ret_series, label=''):
    """Methodology §12.2 metrics from monthly excess return series."""
    r = np.array(ret_series)
    T = len(r)

    ann_ret = (1 + r).prod() ** (12 / T) - 1          # §12.2.1
    ann_vol = r.std() * np.sqrt(12)                    # §12.2.2
    sharpe  = (r.mean() / r.std()) * np.sqrt(12)       # §12.2.3 (r already excess)

    wealth   = (1 + r).cumprod()                       # §12.2.4
    peak     = np.maximum.accumulate(wealth)
    mdd      = ((peak - wealth) / peak).max()

    return {'Portfolio': label, 'Ann. Return': ann_ret,
            'Ann. Vol': ann_vol, 'Sharpe Ratio': sharpe, 'Max Drawdown': mdd}


metrics = [
    performance_metrics(results_ols['ret_opt'], 'OLS + MVO'),
    performance_metrics(results_rf['ret_opt'],  'RF  + MVO'),
    performance_metrics(results_ols['ret_ew'],  'Equal-Weight (OLS stocks)'),
    performance_metrics(results_rf['ret_ew'],   'Equal-Weight (RF stocks)'),
]

perf_table = pd.DataFrame(metrics).set_index('Portfolio')
perf_table_fmt = perf_table.copy()
for col in perf_table_fmt.columns:
    perf_table_fmt[col] = perf_table_fmt[col].map('{:.4f}'.format)

print('\n=== PORTFOLIO PERFORMANCE TABLE (Methodology §13.2) ===')
display(perf_table_fmt)

## 6. Monthly Weight Inspection

In [ ]:
def show_weights(results_df, label, n_months=2):
    rows = []
    for _, row in results_df.iterrows():
        for ticker, w, mu in zip(row['tickers'], row['weights_opt'], row.get('mu_scaled', row['mu'])):
            rows.append({'DateKey': row['DateKey'], 'Ticker': ticker,
                         'Pred Signal': mu, 'Weight': w})
    df = pd.DataFrame(rows)
    sample_months = sorted(df['DateKey'].unique())[:n_months]
    print(f'\n[{label}] Weights — first {n_months} months:')
    display(
        df[df['DateKey'].isin(sample_months)]
        .sort_values(['DateKey', 'Weight'], ascending=[True, False])
        .reset_index(drop=True)
    )

show_weights(results_ols, 'OLS+MVO')
show_weights(results_rf,  'RF+MVO')

## 7. Cumulative Wealth Curves

In [ ]:
months = pd.to_datetime(results_ols['DateKey'].astype(str), format='%Y%m')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── OLS panel ────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(months, (1 + results_ols['ret_opt']).cumprod(),
        label='OLS + MVO (diag Σ)', linewidth=2, color='steelblue')
ax.plot(months, (1 + results_ols['ret_ew']).cumprod(),
        label='Equal-Weight', linewidth=2, linestyle='--', color='orange')
ax.axhline(1, color='grey', linewidth=0.8, linestyle=':')
ax.set_title('OLS Model: MVO vs Equal-Weight')
ax.set_ylabel('Cumulative Wealth ($1 invested)')
ax.legend(); ax.grid(True, alpha=0.3)

# ── RF panel ─────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(months, (1 + results_rf['ret_opt']).cumprod(),
        label='RF + MVO (Ledoit-Wolf Σ)', linewidth=2, color='green')
ax.plot(months, (1 + results_rf['ret_ew']).cumprod(),
        label='Equal-Weight', linewidth=2, linestyle='--', color='orange')
ax.axhline(1, color='grey', linewidth=0.8, linestyle=':')
ax.set_title('RF Model: MVO vs Equal-Weight')
ax.set_ylabel('Cumulative Wealth ($1 invested)')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wealth_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: wealth_curves_v2.png')

## 8. OLS vs RF Head-to-Head

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(months, (1 + results_ols['ret_opt']).cumprod(),
        label='OLS + MVO', linewidth=2, color='steelblue')
ax.plot(months, (1 + results_rf['ret_opt']).cumprod(),
        label='RF  + MVO', linewidth=2, color='green')
ax.plot(months, (1 + results_ols['ret_ew']).cumprod(),
        label='Equal-Weight (OLS stocks)', linewidth=1.5,
        linestyle='--', alpha=0.7, color='orange')
ax.axhline(1, color='grey', linewidth=0.8, linestyle=':')
ax.set_title('Cumulative Wealth: OLS+MVO vs RF+MVO vs Equal-Weight (2022–)')
ax.set_ylabel('Cumulative Wealth')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('head_to_head_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: head_to_head_v2.png')

## 9. Save Results

In [ ]:
monthly_ret = pd.DataFrame({
    'DateKey'       : results_ols['DateKey'].values,
    'OLS_MVO'       : results_ols['ret_opt'].values,
    'RF_MVO'        : results_rf['ret_opt'].values,
    'EW_OLS_stocks' : results_ols['ret_ew'].values,
    'EW_RF_stocks'  : results_rf['ret_ew'].values,
})

monthly_ret.to_csv('mvo_monthly_returns_v2.csv', index=False)
perf_table.to_csv('mvo_performance_summary_v2.csv')

print('Saved: mvo_monthly_returns_v2.csv')
print('Saved: mvo_performance_summary_v2.csv')
display(monthly_ret)